# 실험 01: LLM-as-a-Judge (자체 평가 시스템 구축)

## 1. 실험 제목과 목표
- **제목**: AI가 AI를 채점한다
- **목표**: 외부 평가 프레임워크에 의존하지 않고, 똑똑한 LLM(GPT-4o 등)에게 채점 기준표(프롬프트)를 주어 우리가 만든 RAG 봇의 답변 품질을 스스로 평가하게 만드는 가장 원초적이고 확실한 방법을 실습합니다.

## 2. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

## 3. 환경 변수 로드 및 심사위원(Judge) 준비

In [2]:
load_dotenv()

# 심사위원은 무조건 가장 똑똑한 모델을 쓰는 것이 좋습니다.
# GPT-4o-mini는 답변 생성용, GPT-4o는 심사위원용으로 쓰는 패턴이 흔합니다.
judge_llm = ChatOpenAI(model="gpt-4o-mini") # 실습 환경상 mini를 사용합니다.

## 4. 모의 데이터셋 및 채점 기준(Structured Output) 정의

In [3]:
# 1. 우리가 평가할 가상의 RAG 데이터
question = "우리 회사 환불 규정이 어떻게 돼?"
context = "제품 수령 후 7일 이내에만 환불이 가능하며, 왕복 배송비는 고객 부담입니다."

# 케이스 A: 아주 좋은 봇의 답변
good_answer = "고객님, 저희 회사 환불 규정에 대해 안내해 드리겠습니다. 제품을 수령하신 날로부터 7일 이내에 환불을 요청하실 수 있습니다. 단, 이 과정에서 발생하는 왕복 배송비는 고객님께서 부담하셔야 하는 점 양해 부탁드립니다."

# 케이스 B: 엉뚱한(할루시네이션) 봇의 답변
bad_answer = "환불은 한 달 내내 언제든지 전액 무료로 가능합니다!"

# 2. 심사위원이 뱉어내야 할 '채점표' 규격 정의 (Pydantic)
class EvaluationResult(BaseModel):
    score: int = Field(description="1점에서 10점 사이의 정수 점수. 10점이 만점.")
    reasoning: str = Field(description="이 점수를 주게 된 구체적인 논리적 근거")
    is_hallucination: bool = Field(description="Context(검색된 문서)에 없는 내용을 지어냈다면 True, 아니면 False")

# LLM에 채점표 규격 강제 적용
structured_judge = judge_llm.with_structured_output(EvaluationResult)

## 5. 심사위원 프롬프트 작성

In [4]:
evaluation_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 공정하고 엄격한 AI 답변 심사위원입니다.
    다음 세 가지를 반드시 고려하여 점수를 매기세요:
    1. 답변이 질문에 정확히 대답하고 있는가?
    2. 답변이 주어진 Context(참고 문서)에만 근거하고 있는가? (할루시네이션이 없는가)
    3. 어조가 고객 응대에 적합하게 정중한가?
    """),
    ("human", """[질문]
{question}

[검색된 문서 (Context)]
{context}

[AI가 생성한 답변]
{answer}

위 내용을 바탕으로 평가를 진행해 주세요.""")
])

## 6. 실험 진행 (채점)

In [5]:
def evaluate_bot_answer(answer: str, label: str):
    print(f"\n=== [{label} 평가 중...] ===")
    
    # 프롬프트 완성
    messages = evaluation_prompt.format_messages(
        question=question,
        context=context,
        answer=answer
    )
    
    # 채점 실행
    result = structured_judge.invoke(messages)
    
    print(f"📝 총점: {result.score} / 10")
    print(f"🤔 심사평: {result.reasoning}")
    print(f"🚨 할루시네이션(거짓말) 감지 여부: {result.is_hallucination}")

# 1. 좋은 답변 채점
evaluate_bot_answer(good_answer, "케이스 A (좋은 답변)")

# 2. 나쁜 답변 채점
evaluate_bot_answer(bad_answer, "케이스 B (거짓말하는 답변)")


=== [케이스 A (좋은 답변) 평가 중...] ===
📝 총점: 10 / 10
🤔 심사평: 답변이 질문에 정확히 대답하고 있으며, 주어진 Context에만 근거하고 있습니다. 또한 고객에게 정중한 어조로 안내하고 있어 고객 응대에 적합합니다.
🚨 할루시네이션(거짓말) 감지 여부: False

=== [케이스 B (거짓말하는 답변) 평가 중...] ===
📝 총점: 1 / 10
🤔 심사평: 답변이 질문에 정확히 대답하지 않으며, 주어진 Context에 포함된 정보를 전혀 반영하지 않고 있습니다. 또한, 어조는 정중하나 정보가 잘못된 점이 심각합니다.
🚨 할루시네이션(거짓말) 감지 여부: True


## 7. 결과 해석

1. **LLM-as-a-Judge의 위력**: 사람이 일일이 수천 개의 봇 응답을 읽어보고 평가할 수 없습니다. 대신에 규칙과 프롬프트를 명확히 짠 GPT-4 심사위원을 고용하면, 엄청나게 저렴한 비용으로 봇의 성능을 정량화(수치화)할 수 있습니다.
2. **Pydantic의 결합**: 우리가 이전에 배운 `Structured Output` 기술을 여기서 활용하면, 평가 결과를 딕셔너리나 JSON으로 딱 떨어지게 받을 수 있어서, 나중에 통계(평균 점수 계산 등)를 내기 매우 좋습니다.
3. **한계점**: 이 방식은 프롬프트를 어떻게 쓰느냐에 따라 점수가 들쭉날쭉할 수 있습니다. 그래서 학계와 업계가 합의한 "표준 평가 프레임워크"를 쓰게 되는데, 그것이 바로 다음 노트북에서 배울 **Ragas**입니다.